<a href="https://colab.research.google.com/github/abelunbound/fg_interactive_budget/blob/main/llm_as_a_judge_colab_async.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import os
from dotenv import load_dotenv


In [ ]:
!git clone https://github.com/abelunbound/fg_interactive_budget.git

Cloning into 'fg_interactive_budget'...
remote: Enumerating objects: 258, done.
remote: Counting objects: 100% (258/258), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 258 (delta 146), reused 206 (delta 94), pack-reused 0 (from 0)
Receiving objects: 100% (258/258), 12.78 MiB | 13.27 MiB/s, done.
Resolving deltas: 100% (146/146), done.


In [ ]:
%cd /content/fg_interactive_budget

/content/fg_interactive_budget


In [ ]:
!pwd

In [2]:
# # Mandate data ingestion
input_path_csv = 'https://huggingface.co/datasets/abelakeni/fg-mda-objectives-2026-v1/resolve/main/fg_agencies_objectives_2026_jan.csv'
all_mandates = pd.read_csv(input_path_csv)


In [3]:
print(len(all_mandates))

876


In [4]:
all_mandates.head()

,mda_code,mda_name,mandate_description
0,521025001,"41. COMMUNITY HEALTH TUTOR PROGRAMME, UCH",The Community Health Tutor Programme at the Un...
1,517015001,42. COMPUTER PROFESSIONALS (REGISTRATION COUNC...,The Computer Professionals (Registration Counc...
2,119009118,"43. CONSULATE GENERAL OF NIGERIA, FRANKFURT, G...",The Consulate General of the Federal Republic ...
3,234005001,44. COUNCIL FOR THE REGULATION OF ENGINEERING ...,The Council for the Regulation of Engineering ...
4,229006001,45. COUNCIL FOR THE REGULATION OF FREIGHT FORW...,The Council for the Regulation of Freight Forw...


In [5]:
all_mandates.dtypes

,0
mda_code,int64
mda_name,object
mandate_description,object


In [7]:


# # ergp_budget_with_mda_path = 'https://huggingface.co/datasets/abelakeni/fg-mda-objectives-2026-v1/resolve/main/approved_budget_2026.csv'

# budget_df = pd.read_csv(ergp_budget_with_mda_path, nrows=500)





In [11]:
from huggingface_hub import hf_hub_download
import pandas as pd

# Download the file to a local path
file_path = hf_hub_download(
    repo_id="trackstatecapture/fgbudgetdata",
    filename="DeBERTa_classified_approved_fg_budget_2026.csv",
    repo_type="dataset",
    # token=token  # optional if the dataset is public
)

# Read it with pandas
budget_df = pd.read_csv(file_path, nrows=100)

In [ ]:
budget_df = budget_df.rename(
    columns={
        'mda_name_pdf': 'agency',
        'project': 'ergp_line_item',
        'type': 'status',
    }
)

In [12]:
print(len(budget_df))

100


In [13]:
budget_df.head()

,code,mda_code,agency,ergp_line_item,status,amount,mandate_alignment,classifier_result,predicted_class,prob_outside,prob_within,flag_for_review
0,ERGP26102464,111001001,STATE HOUSE - HQTRS,PURCHASE OF SPORTING EQUIPMENT FOR STATE HOUSE...,ONGOING,8970837,Within MDA Mandate,YES,1,0.000502,0.999498,False
1,ERGP26223957,111001001,STATE HOUSE - HQTRS,PROCUREMENT /MAINTENANCE OF EQUIPMENT FOR THE\...,ONGOING,11713350,Within MDA Mandate,YES,1,0.000386,0.999614,False
2,ERGP27101865,111001001,STATE HOUSE - HQTRS,RENOVATION WORK ON 8 NO. BLOCKS OF 16 NO. 2 B/...,ONGOING,84198642,Within MDA Mandate,YES,1,0.000415,0.999585,False
3,ERGP27201255,111001001,STATE HOUSE - HQTRS,CONSTRUCTION OF OFFICE COMPLEX FOR Sas and SSAs,ONGOING,1281548143,Within MDA Mandate,YES,1,0.000519,0.999481,False
4,ERGP27223750,111001001,STATE HOUSE - HQTRS,ANNUAL ROUTINE MAINTENANCE OF MECHANICAL/ELECT...,ONGOING,4229108872,Within MDA Mandate,YES,1,0.000466,0.999534,False


In [14]:
budget_df.dtypes

,0
code,object
mda_code,int64
agency,object
ergp_line_item,object
status,object
amount,int64
mandate_alignment,object
classifier_result,object
predicted_class,int64
prob_outside,float64


In [ ]:
# Remove commas and convert to float
# budget_df['amount'] = budget_df['amount'].str.replace(',', '').astype('float64')

In [15]:
budget_df.dtypes

,0
code,object
mda_code,int64
agency,object
ergp_line_item,object
status,object
amount,int64
mandate_alignment,object
classifier_result,object
predicted_class,int64
prob_outside,float64


In [16]:
focus_mdas = budget_df['mda_code'].unique().tolist()


In [17]:
print(len(focus_mdas))

10


In [18]:
focus_mdas

[111001001,
 111001002,
 111001003,
 111001004,
 111001005,
 111001006,
 111001007,
 111006001,
 111007001,
 111009001]

In [19]:
focus_mdas_df = all_mandates[all_mandates['mda_code'].isin(focus_mdas)]

In [20]:
print(len(focus_mdas_df))

10


In [21]:
# Install anthropic if needed
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 19.8 MB/s eta 0:00:00


## **Asynchronous - LLM-as-a-Judge**

In [22]:
# =============================================================================
# CONFIGURATION - Add asyncio
# =============================================================================
import asyncio
from anthropic import AsyncAnthropic
import time
from typing import Optional

# Import userdata to access Colab secrets
from google.colab import userdata
import os

ANTHROPIC_API_KEY = userdata.get('CLAUDE_API_KEY')
MODEL_NAME = "claude-sonnet-5"
MAX_RETRIES = 3
RETRY_DELAY = 2
MAX_CONCURRENT_REQUESTS = 10  # Control concurrency to avoid rate limits

In [23]:
# =============================================================================
# CORE FUNCTIONS - with async processing
# =============================================================================

async def judge_alignment_async(
    client: AsyncAnthropic,
    line_item: str,
    agency: str,
    mandate: str,
    semaphore: asyncio.Semaphore
) -> dict:
    """Async version of judge_alignment."""
    prompt = f"""You are an auditor checking if Nigerian government budget items align with agency mandates.

    AGENCY: {agency}

    AGENCY MANDATE: {mandate}

    BUDGET LINE ITEM: {line_item}

    Does this budget item fall within the agency's mandate?

    Respond in EXACTLY this format (no other text):
    ALIGNMENT: YES or PARTIAL or NO
    CONFIDENCE: HIGH or MEDIUM or LOW
    REASON: (one sentence explanation)"""

    async with semaphore:  # Limit concurrent requests
        for attempt in range(MAX_RETRIES):
            try:
                response = await client.messages.create(
                    model=MODEL_NAME,
                    max_tokens=150,
                    messages=[{"role": "user", "content": prompt}]
                )

                result_text = response.content[0].text
                return parse_llm_response(result_text)

            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    await asyncio.sleep(RETRY_DELAY)
                else:
                    return {
                        "alignment": "ERROR",
                        "confidence": "UNKNOWN",
                        "reason": str(e)
                    }



In [24]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def parse_llm_response(response: str) -> dict:
    """Parse the structured LLM response."""
    result = {
        "alignment": "UNKNOWN",
        "confidence": "UNKNOWN",
        "reason": ""
    }

    lines = response.strip().split("\n")

    for line in lines:
        line = line.strip()
        if line.upper().startswith("ALIGNMENT:"):
            value = line.split(":", 1)[1].strip().upper()
            if value in ["YES", "PARTIAL", "NO"]:
                result["alignment"] = value
        elif line.upper().startswith("CONFIDENCE:"):
            value = line.split(":", 1)[1].strip().upper()
            if value in ["HIGH", "MEDIUM", "LOW"]:
                result["confidence"] = value
        elif line.upper().startswith("REASON:"):
            result["reason"] = line.split(":", 1)[1].strip()

    return result


def compute_deviation_score(alignment: str, confidence: str) -> float:
    """
    Convert alignment + confidence to a deviation score (0-100).

    Higher score = more likely outside mandate = higher audit priority.
    """
    alignment_scores = {
        "YES": 0,
        "PARTIAL": 50,
        "NO": 100,
        "UNKNOWN": 50,
        "ERROR": 50
    }

    confidence_adjustments = {
        "HIGH": 0,
        "MEDIUM": 5,
        "LOW": 10,
        "UNKNOWN": 10
    }

    base_score = alignment_scores.get(alignment, 50)
    adjustment = confidence_adjustments.get(confidence, 10)

    # For YES: add adjustment (less confident YES = slightly higher deviation)
    # For NO: subtract adjustment (less confident NO = slightly lower deviation)
    if alignment == "YES":
        return min(base_score + adjustment, 100)
    elif alignment == "NO":
        return max(base_score - adjustment, 0)
    else:
        return base_score

In [25]:
# =============================================================================
# MAIN ANALYSIS FUNCTION - FIXED
# =============================================================================

async def analyze_budget_mandate_alignment_llm_async(
    budget_df: pd.DataFrame,
    mandate_df: pd.DataFrame,
    api_key: str = ANTHROPIC_API_KEY,
    max_concurrent: int = MAX_CONCURRENT_REQUESTS,
    progress_interval: int = 10
) -> pd.DataFrame:
    """Async version - processes requests concurrently."""

    client = AsyncAnthropic(api_key=api_key)
    mandate_lookup = dict(zip(mandate_df["mda_code"], mandate_df["mandate_description"]))

    semaphore = asyncio.Semaphore(max_concurrent)
    total_items = len(budget_df)

    print(f"Analyzing {total_items} budget items with Claude (async)...")
    print(f"Estimated time: {total_items * 2.5 / 60 / max_concurrent:.1f} - {total_items * 3 / 60 / max_concurrent:.1f} minutes")
    print("=" * 60)

    start_time = time.time()

    # Helper coroutine to bundle row with its result
    async def process_row_with_data(row, task):
        """Process task and return (row, judgement) tuple."""
        judgement = await task
        return (row, judgement)

    # Create all tasks bundled with row data
    bundled_tasks = []
    for _, row in budget_df.iterrows():
        mda_code = row["mda_code"]
        mandate = mandate_lookup.get(mda_code, "")

        if mandate:
            task = judge_alignment_async(
                client,
                row["ergp_line_item"],
                row["agency"],
                mandate,
                semaphore
            )
        else:
            async def no_mandate():
                return {
                    "alignment": "UNKNOWN",
                    "confidence": "UNKNOWN",
                    "reason": "No mandate found for this agency"
                }
            task = no_mandate()

        # Bundle row with task
        bundled = process_row_with_data(row, task)
        bundled_tasks.append(bundled)

    # Process results as they complete
    results = []
    completed_count = 0

    for coro in asyncio.as_completed(bundled_tasks):
        row, judgement = await coro  # ← Now we get the CORRECT row for this result!

        deviation = compute_deviation_score(
            judgement["alignment"],
            judgement["confidence"]
        )

        results.append({
            "code": row["code"],
            "ergp_line_item": row["ergp_line_item"],
            "mda_code": row["mda_code"],
            "agency": row["agency"],
            "amount": row["amount"],
            "alignment": judgement["alignment"],
            "confidence": judgement["confidence"],
            "reason": judgement["reason"],
            "mandate_deviation_score": deviation
        })

        completed_count += 1

        # Progress update
        if completed_count % progress_interval == 0 or completed_count == total_items:
            elapsed = time.time() - start_time
            rate = completed_count / elapsed
            remaining = (total_items - completed_count) / rate if rate > 0 else 0
            print(f"   Processed {completed_count}/{total_items} ({completed_count/total_items*100:.1f}%) | "
                  f"Elapsed: {elapsed/60:.1f}min | Remaining: {remaining/60:.1f}min")

    print("=" * 60)
    print("Analysis complete.")

    return pd.DataFrame(results)

In [26]:
# =============================================================================
# OUTPUT & REPORTING
# =============================================================================

def print_summary_llm(result_df: pd.DataFrame) -> None:
    """Print summary statistics for LLM analysis."""
    print("\n" + "=" * 80)
    print("SUMMARY: LLM-AS-JUDGE ANALYSIS")
    print("=" * 80)

    total_items = len(result_df)
    total_amount = result_df["amount"].sum()

    # Alignment distribution
    alignment_counts = result_df["alignment"].value_counts()

    print(f"\nTotal Budget Items: {total_items}")
    print(f"Total Budget Amount: ₦{total_amount:,.0f}")

    print(f"\nAlignment Distribution:")
    for alignment, count in alignment_counts.items():
        pct = count / total_items * 100
        amount = result_df[result_df["alignment"] == alignment]["amount"].sum()
        print(f"   {alignment}: {count} items ({pct:.1f}%) | ₦{amount:,.0f}")

    # Flagged items (NO alignment)
    flagged = result_df[result_df["alignment"] == "NO"]
    print(f"\n🚨 Items flagged as NOT aligned: {len(flagged)}")
    print(f"   Flagged amount: ₦{flagged['amount'].sum():,.0f}")


def print_flagged_items_llm(result_df: pd.DataFrame, top_n: int = 20) -> None:
    """Print items flagged as not aligned."""
    flagged = result_df[result_df["alignment"] == "NO"].sort_values(
        "amount", ascending=False
    ).head(top_n)

    if len(flagged) == 0:
        print("\n✅ No items flagged as misaligned!")
        return

    print("\n" + "=" * 100)
    print(f"TOP {top_n} FLAGGED ITEMS (NOT ALIGNED)")
    print("=" * 100)

    for _, row in flagged.iterrows():
        print(f"\n🚨 {row['code']}: {row['ergp_line_item'][:60]}...")
        print(f"   Agency: {row['agency']}")
        print(f"   Amount: ₦{row['amount']:,.0f}")
        print(f"   Confidence: {row['confidence']}")
        print(f"   Reason: {row['reason']}")



In [27]:
# =============================================================================
# USAGE - Run with asyncio
# =============================================================================

# 1. Set your API key
api_key = ANTHROPIC_API_KEY

# Run the async function
async_v2_result_df = await analyze_budget_mandate_alignment_llm_async(
    budget_df=budget_df,
    mandate_df=focus_mdas_df,
    api_key=api_key
)

Analyzing 100 budget items with Claude (async)...
Estimated time: 0.4 - 0.5 minutes
   Processed 10/100 (10.0%) | Elapsed: 0.1min | Remaining: 0.5min
   Processed 20/100 (20.0%) | Elapsed: 0.1min | Remaining: 0.4min
   Processed 30/100 (30.0%) | Elapsed: 0.1min | Remaining: 0.3min
   Processed 40/100 (40.0%) | Elapsed: 0.2min | Remaining: 0.2min
   Processed 50/100 (50.0%) | Elapsed: 0.2min | Remaining: 0.2min
   Processed 60/100 (60.0%) | Elapsed: 0.2min | Remaining: 0.2min
   Processed 70/100 (70.0%) | Elapsed: 0.3min | Remaining: 0.1min
   Processed 80/100 (80.0%) | Elapsed: 0.3min | Remaining: 0.1min
   Processed 90/100 (90.0%) | Elapsed: 0.3min | Remaining: 0.0min
   Processed 100/100 (100.0%) | Elapsed: 0.5min | Remaining: 0.0min
Analysis complete.


In [29]:
# 3. View results
print_summary_llm(async_v2_result_df)
print_flagged_items_llm(async_v2_result_df)




SUMMARY: LLM-AS-JUDGE ANALYSIS

Total Budget Items: 100
Total Budget Amount: ₦65,856,175,977

Alignment Distribution:
   YES: 69 items (69.0%) | ₦48,173,840,656
   PARTIAL: 25 items (25.0%) | ₦2,917,280,014
   NO: 5 items (5.0%) | ₦14,391,161,022
   ERROR: 1 items (1.0%) | ₦373,894,285

🚨 Items flagged as NOT aligned: 5
   Flagged amount: ₦14,391,161,022

TOP 20 FLAGGED ITEMS (NOT ALIGNED)

🚨 ERGP16234836: "ACQUISITION OF BUREAU OF PUBLIC PROCUREMENT
HEADQUARTER OFF...
   Agency: BUREAU OF PUBLIC ENTERPRISES (BPE)
   Amount: ₦14,000,000,000
   Confidence: HIGH
   Reason: This item concerns acquiring an office headquarters for the Bureau of Public Procurement, a separate agency, and is unrelated to BPE's privatisation and commercialisation mandate.

🚨 ERGP23234834: EMPOWERING OF WOMEN & YOUTH IN ORLU-ORSU-ORU
EAST FEDERAL CO...
   Agency: NIPSS, KURU
   Amount: ₦143,226,237
   Confidence: HIGH
   Reason: This constituency-specific women and youth empowerment project is a constituency d

In [ ]:
# =============================================================================
# USAGE - EXPERIMENT 3 - Run with asyncio
# =============================================================================

# 1. Set your API key
api_key = ANTHROPIC_API_KEY


# 2. Run the async function
async_v4_result_df = await analyze_budget_mandate_alignment_llm_async(
    budget_df=budget_df,
    mandate_df=focus_mdas_df,
    api_key=api_key
)

Analyzing 500 budget items with Claude (async)...
Estimated time: 2.1 - 2.5 minutes
   Processed 10/500 (2.0%) | Elapsed: 0.1min | Remaining: 3.4min
   Processed 20/500 (4.0%) | Elapsed: 0.1min | Remaining: 2.7min
   Processed 30/500 (6.0%) | Elapsed: 0.2min | Remaining: 2.4min
   Processed 40/500 (8.0%) | Elapsed: 0.2min | Remaining: 2.3min
   Processed 50/500 (10.0%) | Elapsed: 0.2min | Remaining: 2.2min
   Processed 60/500 (12.0%) | Elapsed: 0.3min | Remaining: 2.1min
   Processed 70/500 (14.0%) | Elapsed: 0.3min | Remaining: 2.1min
   Processed 80/500 (16.0%) | Elapsed: 0.4min | Remaining: 2.0min
   Processed 90/500 (18.0%) | Elapsed: 0.4min | Remaining: 1.9min
   Processed 100/500 (20.0%) | Elapsed: 0.5min | Remaining: 1.9min
   Processed 110/500 (22.0%) | Elapsed: 0.5min | Remaining: 1.8min
   Processed 120/500 (24.0%) | Elapsed: 0.6min | Remaining: 1.8min
   Processed 130/500 (26.0%) | Elapsed: 0.6min | Remaining: 1.7min
   Processed 140/500 (28.0%) | Elapsed: 0.6min | Remaining

In [ ]:
# =============================================================================
# USAGE 3. View results
# =============================================================================



print_summary_llm(async_v4_result_df)
print_flagged_items_llm(async_v4_result_df)




SUMMARY: LLM-AS-JUDGE ANALYSIS

Total Budget Items: 500
Total Budget Amount: ₦257,496,574,655

Alignment Distribution:
   NO: 225 items (45.0%) | ₦103,830,645,700
   YES: 170 items (34.0%) | ₦139,085,402,572
   PARTIAL: 105 items (21.0%) | ₦14,580,526,383

🚨 Items flagged as NOT aligned: 225
   Flagged amount: ₦103,830,645,700

TOP 20 FLAGGED ITEMS (NOT ALIGNED)

🚨 ERGP20269504: INFRASTRUCTURE DEVELOPMENT...
   Agency: BUREAU OF PUBLIC PROCUREMENT (BPP)
   Amount: ₦20,000,000,000
   Confidence: HIGH
   Reason: The BPP's mandate is strictly regulatory and oversight-focused — governing procurement processes, standards, and compliance — not the direct development or funding of physical infrastructure, which falls under the mandates of construction or sectoral agencies.

🚨 ERGP16234836: "ACQUISITION OF BUREAU OF PUBLIC PROCUREMENT
HEADQUARTER OFF...
   Agency: BUREAU OF PUBLIC ENTERPRISES (BPE)
   Amount: ₦14,000,000,000
   Confidence: HIGH
   Reason: This budget item refers to the Bureau

In [ ]:
# 4. Export
async_v4_result_df.to_csv("/content/async_v5_mandate_alignment_500_test.csv", index=False)

pass